In [38]:
import pandas as pd

In [39]:
data_inp = pd.read_excel("Hospital Operations Dashboard Sample Data - FINAL.xlsx", sheet_name='Inpatients')
data_ED = pd.read_excel("Hospital Operations Dashboard Sample Data - FINAL.xlsx", sheet_name='ED')
data_ALC = pd.read_excel("Hospital Operations Dashboard Sample Data - FINAL.xlsx", sheet_name='ALC')


# Inpatients

In [40]:
# Step 1: Extract relevant blocks
inpatient_beds_block = data_inp.iloc[1:6, :]  # Rows for the 5 sites

separations_block = data_inp.iloc[9:14, :]

unfunded_beds_block = data_inp.iloc[17:22, :]

In [41]:
# Step 2: Drop the Min and Max columns from each block
inpatient_beds_block = inpatient_beds_block.drop(columns=['Min', 'Max'])
separations_block = separations_block.drop(columns=['Min', 'Max'])
unfunded_beds_block = unfunded_beds_block.drop(columns=['Min', 'Max'])

In [42]:
# Step 3: Reshape each block into long format
inpatient_beds_long = inpatient_beds_block.melt(id_vars=['Day'], var_name='Day_Number', value_name='Inpatient Bed Count')
separations_long = separations_block.melt(id_vars=['Day'], var_name='Day_Number', value_name='Separations')
unfunded_beds_long = unfunded_beds_block.melt(id_vars=['Day'], var_name='Day_Number', value_name='Unfunded Beds')

In [43]:
# Step 4: Merge the reshaped blocks
final_inp_data = pd.merge(inpatient_beds_long, separations_long[['Day', 'Day_Number', 'Separations']], on=['Day', 'Day_Number'])
final_inp_data = pd.merge(final_inp_data, unfunded_beds_long[['Day', 'Day_Number', 'Unfunded Beds']], on=['Day', 'Day_Number'])

In [44]:
# Step 5: Rename 'Day' to 'Site' for clarity
final_inp_data.rename(columns={'Day': 'Site'}, inplace=True)

## Add Admission Column

In [45]:
# Step 6: Calculate Admission for each day per site
# Admission = Inpatient Bed Count(current day) - Inpatient Bed Count(previous day) + Separations(current day)
final_inp_data['Previous Inpatient Bed Count'] = final_inp_data.groupby(['Site'])['Inpatient Bed Count'].shift(1)
final_inp_data['Admission'] = final_inp_data['Inpatient Bed Count'] - final_inp_data['Previous Inpatient Bed Count'].fillna(0) + final_inp_data['Separations']
final_inp_data = final_inp_data.drop(columns='Previous Inpatient Bed Count')

## Checking Missing Values

In [46]:
final_inp_data.isna().sum()

Site                   0
Day_Number             0
Inpatient Bed Count    0
Separations            0
Unfunded Beds          0
Admission              0
dtype: int64

# ED

In [47]:
# Step 1: Extract relevant blocks
# ED Total Visit is at the beginning (assuming rows 1 to 5 for the 5 sites)
ed_visit_block = data_ED.iloc[1:6, :]  # Rows for the 5 sites

# ED Admissions Rates start from row 9 for 5 sites (similar pattern as Inpatients)
admissions_rate_block = data_ED.iloc[9:14, :]

# ED Admit (no bed) start from row 17 for 5 sites
ed_admit_no_bed_block = data_ED.iloc[17:22, :]

# Step 2: Drop the Min and Max columns from each block
ed_visit_block = ed_visit_block.drop(columns=['Min', 'Max'])
admissions_rate_block = admissions_rate_block.drop(columns=['Min', 'Max'])
ed_admit_no_bed_block = ed_admit_no_bed_block.drop(columns=['Min', 'Max'])

# Step 3: Reshape each block into long format
ed_visit_long = ed_visit_block.melt(id_vars=['Day'], var_name='Day_Number', value_name='ED Visits')
admissions_rate_long = admissions_rate_block.melt(id_vars=['Day'], var_name='Day_Number', value_name='ED Admission Rate')
ed_admit_no_bed_long = ed_admit_no_bed_block.melt(id_vars=['Day'], var_name='Day_Number', value_name='ED Admit (no bed)')

# Step 4: Merge the reshaped blocks
final_ed_data = pd.merge(ed_visit_long, admissions_rate_long[['Day', 'Day_Number', 'ED Admission Rate']], on=['Day', 'Day_Number'])
final_ed_data = pd.merge(final_ed_data, ed_admit_no_bed_long[['Day', 'Day_Number', 'ED Admit (no bed)']], on=['Day', 'Day_Number'])

# Step 5: Rename 'Day' to 'Site' for clarity
final_ed_data.rename(columns={'Day': 'Site'}, inplace=True)


## Checking Missing Values

In [48]:
final_ed_data.isna().sum()

Site                 0
Day_Number           0
ED Visits            0
ED Admission Rate    0
ED Admit (no bed)    2
dtype: int64

In [49]:
# Step 1: Group by 'Site' and calculate the mean of 'Admit (no bed)' for each site
mean_admit_no_bed_by_site = final_ed_data.groupby('Site')['ED Admit (no bed)'].transform('mean')

# Step 2: Impute missing values (NaN) in 'Admit (no bed)' with the mean of that column for the associated site
final_ed_data['ED Admit (no bed)'] = final_ed_data['ED Admit (no bed)'].fillna(mean_admit_no_bed_by_site)



In [50]:
final_ed_data.iloc[50:60]

,Site,Day_Number,ED Visits,ED Admission Rate,ED Admit (no bed)
50,Site #1,11,111.0,0.252252,10.000000
51,Site #2,11,102.0,0.235294,11.000000
52,Site #3,11,141.0,0.106383,4.241483
53,Site #4,11,61.0,0.000000,0.000000
54,Site #5,11,59.0,0.033898,1.000000
55,Site #1,12,145.0,0.186207,14.000000
56,Site #2,12,111.0,0.171171,14.000000
57,Site #3,12,179.0,0.089385,4.241483
58,Site #4,12,105.0,0.000000,0.000000
59,Site #5,12,43.0,0.069767,1.000000


In [51]:
final_ed_data.isna().sum()

Site                 0
Day_Number           0
ED Visits            0
ED Admission Rate    0
ED Admit (no bed)    0
dtype: int64

# ALC

In [52]:
# Step 1: Extract relevant rows (for Sites and Designations), including TOTAL
alc_count_block = data_ALC[data_ALC['Day'].str.contains('Designation|Site|TOTAL', na=False)]

# Step 2: Drop the Min and Max columns
alc_count_block = alc_count_block.drop(columns=['Min', 'Max'])

# Step 3: Create separate columns for Site and Designation
alc_count_block['Site'] = alc_count_block['Day'].apply(lambda x: x if 'Site' in x or 'TOTAL' in x else None)
alc_count_block['Designation'] = alc_count_block['Day'].apply(lambda x: x if 'Designation' in x else None)

# Step 4: Reshape the ALC Count block into long format
final_alc_data = alc_count_block.melt(id_vars=['Site', 'Designation'], var_name='Day_Number', value_name='ALC Count')

# Step 5: Convert the 'Day_Number' column to numeric and drop non-numeric values
final_alc_data['Day_Number'] = pd.to_numeric(final_alc_data['Day_Number'], errors='coerce')
final_alc_data = final_alc_data.dropna(subset=['Day_Number'])

# Step 6: Repeat site names where applicable (fill down the "Site" column)
final_alc_data['Site'] = final_alc_data['Site'].ffill()

# Step 7: Remove rows where Designation is empty
final_alc_data = final_alc_data.dropna(subset=['Designation'])

# Step 8: Remove the string "ALC Count - " from the Site column
final_alc_data['Site'] = final_alc_data['Site'].str.replace('ALC Count - ', '', regex=False)

## Standardized Site Column

In [53]:
final_alc_data['Site'].unique()

array(['TOTAL', 'Site 1', 'Site 2', 'Site 3'], dtype=object)

In [54]:
# Mapping dictionary
site_mapping = {
    'Site 1': 'Site #1',
    'Site 2': 'Site #2',
    'Site 3': 'Site #3',
    'TOTAL': 'TOTAL'
}

# Apply the mapping to the 'Site' column
final_alc_data['Standardized Site'] = final_alc_data['Site'].replace(site_mapping)

In [55]:
# Assuming 'ALC Count' is currently of type 'object'
final_alc_data['ALC Count'] = pd.to_numeric(final_alc_data['ALC Count'], errors='coerce')

## Checking Missing Values

In [56]:
final_alc_data.isna().sum()

Site                 0
Designation          0
Day_Number           0
ALC Count            0
Standardized Site    0
dtype: int64

## Discrepancy

In [57]:
final_alc_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21000 entries, 26 to 25024
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Site               21000 non-null  object 
 1   Designation        21000 non-null  object 
 2   Day_Number         21000 non-null  float64
 3   ALC Count          21000 non-null  float64
 4   Standardized Site  21000 non-null  object 
dtypes: float64(2), object(3)
memory usage: 984.4+ KB


In [58]:
# Step 1: Calculate the sum of "ALC Count" for "Site 1", "Site 2", and "Site 3" for each "Day_Number" and "Designation"
site_sum = final_alc_data[final_alc_data['Site'].isin(['Site 1', 'Site 2', 'Site 3'])].groupby(['Day_Number', 'Designation'])['ALC Count'].sum().reset_index(name='Site_Sum_ALC')

# Step 2: Get the "ALC Count" for "TOTAL" for each "Day_Number" and "Designation"
total_data = final_alc_data[final_alc_data['Site'] == 'TOTAL'][['Day_Number', 'Designation', 'ALC Count']].rename(columns={'ALC Count': 'Total_ALC'})

# Step 3: Merge the two dataframes on "Day_Number" and "Designation"
comparison_df = pd.merge(site_sum, total_data, on=['Day_Number', 'Designation'])

# Step 4: Compare the sum of "Site 1", "Site 2", and "Site 3" to "TOTAL"
comparison_df['Comparison'] = comparison_df['Site_Sum_ALC'] <= comparison_df['Total_ALC']

comparison_df['Difference'] = comparison_df['Total_ALC'] - comparison_df['Site_Sum_ALC']

In [59]:
# Step 5: Filter rows where the sum is not less than or equal to the total (False cases)
false_comparisons = comparison_df[~comparison_df['Comparison']]

# Step 6: Get the unique values in the "Designation" column and the shape of the result
unique_designations = false_comparisons['Designation'].unique()

# Output results
print("Unique Designations in False Cases:", unique_designations)

Unique Designations in False Cases: ['Designation #1' 'Designation Other' 'Designation #4' 'Designation #5']


In [60]:
average_difference_F = false_comparisons.groupby('Designation')['Difference'].mean().reset_index()
average_difference_F

,Designation,Difference
0,Designation #1,-20.316000
1,Designation #4,-2.023256
2,Designation #5,-2.254777
3,Designation Other,-4.301587


In [61]:
true_comparisons = comparison_df[comparison_df['Comparison']]

average_difference_T = true_comparisons.groupby('Designation')['Difference'].mean().reset_index()
average_difference_T

,Designation,Difference
0,Designation #2,12.302000
1,Designation #3,6.823000
2,Designation #4,3.830918
3,Designation #5,3.956109
4,Designation Other,3.749553


In [62]:
false_comparisons['Designation'].value_counts()

Designation #1       1000
Designation Other     441
Designation #4        172
Designation #5        157
Name: Designation, dtype: int64

# Export

In [26]:
# with pd.ExcelWriter('ReshapedData.xlsx',engine='xlsxwriter') as writer:
#     final_inp_data.to_excel(writer, sheet_name='Inpatients Data', index=False)
#     final_alc_data.to_excel(writer, sheet_name='ALC Data', index=False)
#     final_ed_data.to_excel(writer, sheet_name='ED Data', index=False)